<a href="https://colab.research.google.com/github/CVelo-A/03MIAR_Algoritmos_Optimizacion/blob/main/Algoritmos_Trabajo_Pr%C3%A1ctico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Trabajo Práctico<br>
Nombre y Apellidos: Christian Velo Abella  <br>
Url: https://github.com/CVelo-A/03MIAR_Algoritmos_Optimizacion.git<br>
Google Colab: https://colab.research.google.com/drive/1Q6cjzlMSGIgQJb9btjectcMZsqLZw0sk?usp=sharing <br>

#Problema 2.
##Apartado(I).

• Desde la La Liga de fútbol profesional se pretende organizar los horarios de los partidos de
liga de cada jornada. Se conocen algunos datos que nos deben llevar a diseñar un
algoritmo que realice la asignación de los partidos a los horarios de forma que maximice la audiencia.

Los horarios disponibles se conocen a priori y son los siguientes:

| Dia | Horario   |
|---------|------|
| Viernes | 20   |
| Sábado  | 12, 16, 18, 20 |
| Domingo | 12, 16, 18, 20 |
| Lunes   | 20   |

##Apartado(II)

• En primer lugar se clasifican los equipos en tres categorías según el numero de seguidores (que tiene relación directa con la audiencia). Hay 3 equipos en la
categoría A, 11 equipos de categoría B y 6 equipos de categoría C.

• Se conoce estadísticamente la audiencia que genera cada partido según los quipos que se enfrentan y en horario de sábado a las 20h (el mejor en todos los casos)

|             | Categoría A | Categoría B | Categoría C |
|-------------|-------------|-------------|-------------|
| **Categoría A** | 2 Millones  | 1,3 Millones | 1 Millón    |
| **Categoría B** | 0.9 Millones| 0.75 Millones|             |
| **Categoría C** | 0.47 Millones|             |             |

##Apartado(III)
• Si el horario del partido no se realiza a las 20 horas del sábado se sabe que se reduce según los coeficientes de la siguiente tabla
• Debemos asignar obligatoriamente siempre un partido el viernes y un partido el lunes

| Hora  | Viernes | Sábado | Domingo | Lunes  |
|-------|---------|--------|---------|--------|
| 12h   | -    | 0.55   | 0.45       | -      |
| 16h   |  -    |  0.7  | 0.75       | -      |
| 18h   |  -   |   0.8  | 0.85       | -      |
| 20h   | 0.4     | 1      | 1       | 0.4    |

##Apartado(IV)
• Es posible la coincidencia de horarios pero en este caso la audiencia de cada partido se verá afectada y se estima que se reduce en porcentaje según la siguiente tabla dependiendo del número de coincidencias:

| Coincidencia | -%   | Coincidencia | -%   |
|---------|------|---------|------|
| 0 | 0%  | 5 | 75%  |
| 1  | 25%  | 6  | 78%  |
| 2 |  45%  | 7 |  80%  |
| 3   | 60%  | 8   | 80%  |
| 4  | 70%  |

                                        

#Modelo
- ¿Como represento el espacio de soluciones?
- ¿Cual es la función objetivo?
- ¿Como implemento las restricciones?

El espacio de soluciones esta representado por una asignacion de horarios a enfrentamiento de partidos de la jornada propuesta. Teniendo en la lista de enfrentamiento en el indice 0 y 1 los equipos y en 2 y 3 la categoria de cada uno de los anteriores.

La funcion objetivo es el sumatorio de las audiencias base de las dos categorias de equipos que componen cada encuetro multiplicado por el coeficiente de reduccion por horario y la penalizacion de coincidencia. El objetivo es maximizar la audiencia en la jornada.

Se itera sobre las diferentes soluciones y previamente a su analisis con anteriores soluciones se aplica las reducciones/penalizaciones con la utilizacion de coef_reduccion y tambien con la reduccion_coincidencia.




#Análisis
- ¿Que complejidad tiene el problema?. Orden de complejidad y Contabilizar el espacio de soluciones

En la primera parte del problema nos encontramos con complejidad O(n) debido a que la parte mas restrictiva es el shuffle y el while, ambos con O(n).
En la segunda parte nos encontramos con que la generacion de vecinos nos impone una complejidad O(n^2) que posteriormente, y al evaluarse la audiencia se incrementa a O(n^3).
Por lo tanto la complejidad del problema se estima en O(n^3), impuesto por esta ultima parte del problema.

#Diseño
- ¿Que técnica utilizo? ¿Por qué?

Utilizo la busqueda Tabu ya que evaluar todas las posibles combinaciones seria muy costoso computacionalmente, asimismo me permite explorar "malas" soluciones para eviar quedar atrapado en optimo local.

### Datos iniciales y generacion de jornada a asignar.

In [6]:
import random

# Equipos y categoria.
equipos = {
    'A': ['A1', 'A2', 'A3'],
    'B': ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B9', 'B10', 'B11'],
    'C': ['C1', 'C2', 'C3', 'C4', 'C5', 'C6']
}
# Datos audiencia base.
audiencia_base = {
    ('A', 'A'): 2.0, ('A', 'B'): 1.3, ('A', 'C'): 1.0,
    ('B', 'B'): 0.9, ('B', 'C'): 0.75,
    ('C', 'C'): 0.47
}
# Coeficientes reduccion por horario.
coef_reduccion = {
    'Viernes 20': 0.4, 'Sabado 12': 0.55, 'Sabado 16': 0.7,
    'Sabado 18': 0.8, 'Sabado 20': 1.0, 'Domingo 12': 0.45,
    'Domingo 16': 0.75, 'Domingo 18': 0.85, 'Domingo 20': 1.0,
    'Lunes 20': 0.4
}
# Penalizacion de coincidencia.
reduccion_coincidencia = [0, 0.25, 0.45, 0.60, 0.70, 0.75, 0.78, 0.80, 0.80]
# Horarios disponibles.
horarios = ['Viernes 20', 'Sabado 12', 'Sabado 16', 'Sabado 18', 'Sabado 20',
            'Domingo 12', 'Domingo 16', 'Domingo 18', 'Domingo 20', 'Lunes 20']
numero_horarios = len(horarios)

# Obtencion de la categoria.
def obtener_categoria(equipo):
    for categoria, lista_equipos in equipos.items():
        if equipo in lista_equipos:
            return categoria
    return None

# Generar jornada aleatoriamente con semilla random.
def generar_jornada():
    random.seed(10)
    equipos_disponibles = equipos['A'] + equipos['B'] + equipos['C']
    random.shuffle(equipos_disponibles)
    partidos = []

    while len(equipos_disponibles) >= 2:
        equipo1 = equipos_disponibles.pop()
        equipo2 = equipos_disponibles.pop()
        cat1 = obtener_categoria(equipo1)
        cat2 = obtener_categoria(equipo2)
        partidos.append((equipo1, equipo2, cat1, cat2))

    return partidos

# Generar jornada aleatoria.
partidos = generar_jornada()
print("La jornada generada para asignar horarios es:")
for e, p in enumerate(partidos):
    print(f'Encuentro {e+1}: {p[0]} vs {p[1]}')


La jornada generada para asignar horarios es:
Encuentro 1: C5 vs A2
Encuentro 2: B11 vs C2
Encuentro 3: A1 vs B1
Encuentro 4: B5 vs C4
Encuentro 5: B2 vs B8
Encuentro 6: A3 vs C3
Encuentro 7: B10 vs B7
Encuentro 8: B6 vs C6
Encuentro 9: B4 vs B3
Encuentro 10: B9 vs C1


### Desarrollo de codigo de encuadre de partidos.

In [38]:
# Calcular audiencia.
def calcular_audiencia(solucion):
    audiencia_total = 0
    horarios_ocupados = [[] for _ in range(numero_horarios)]

    for i, horario in enumerate(solucion):
        partido = partidos[i]
        cat1, cat2 = partido[2], partido[3]

        if (cat1, cat2) not in audiencia_base:
            cat1, cat2 = sorted((cat1, cat2))

        base = audiencia_base[(cat1, cat2)]
        coef = coef_reduccion[horarios[horario]]
        audiencia_total += base * coef
        horarios_ocupados[horario].append(i)

    # Penalizacion de coincidencias.
    for h in horarios_ocupados:
        num_partidos = len(h)
        if num_partidos > 1:
            penalizacion = reduccion_coincidencia[min(num_partidos - 1, len(reduccion_coincidencia) - 1)]
            audiencia_total *= (1 - penalizacion)
    return audiencia_total

# Generar vecinos.
def generar_vecinos(solucion):
    vecinos = []
    for i in range(len(solucion)):
        for j in range(i + 1, len(solucion)):
            vecino = solucion[:]
            vecino[i], vecino[j] = vecino[j], vecino[i]
            vecinos.append(vecino)
    return vecinos

# Busqueda TABU.
def busqueda_tabu(iteraciones=100, dim_tabu = 10):
    solucion_actual = list(range(len(partidos)))
    random.shuffle(solucion_actual)
    mejor_solucion = solucion_actual[:]
    mejor_audiencia = calcular_audiencia(mejor_solucion)
    memoria_tabu = []

    print(f'Solución inicial: {solucion_actual}, Audiencia: {mejor_audiencia:.2f}\n')

    for iteracion in range(iteraciones):
        vecinos = generar_vecinos(solucion_actual)
        mejor_vecino = None
        mejor_audiencia_vecino = -float('inf')

        for vecino in vecinos:
            if vecino not in memoria_tabu:
                audiencia_vecino = calcular_audiencia(vecino)
                if audiencia_vecino > mejor_audiencia_vecino:
                    mejor_audiencia_vecino = audiencia_vecino
                    mejor_vecino = vecino
        if mejor_vecino is None:
            break

        solucion_actual = mejor_vecino
        if mejor_audiencia_vecino > mejor_audiencia:
            mejor_solucion = mejor_vecino[:]
            mejor_audiencia = mejor_audiencia_vecino
            print(f'Iteracion {iteracion + 1}: Nueva mejor audiencia {mejor_audiencia:.2f}')

        memoria_tabu.append(mejor_vecino)
        if len(memoria_tabu) > dim_tabu:
            memoria_tabu.pop(0)

    return mejor_solucion, mejor_audiencia




# Optimizacion de la jornada.
solucion, audiencia = busqueda_tabu()
print('\nMejor solucion encontrada:')
for p, horario in zip(partidos, solucion):
    print(f'Equipo {p[0]} vs Equipo {p[1]}--> {horarios[horario]}')
print(f'\nAudiencia Total: {audiencia:.2f} millones')


Solución inicial: [1, 7, 6, 5, 3, 2, 4, 8, 9, 0], Audiencia: 6.23

Iteracion 1: Nueva mejor audiencia 6.37
Iteracion 2: Nueva mejor audiencia 6.44
Iteracion 3: Nueva mejor audiencia 6.50
Iteracion 4: Nueva mejor audiencia 6.52
Iteracion 6: Nueva mejor audiencia 6.53

Mejor solucion encontrada:
Equipo C5 vs Equipo A2--> Sabado 20
Equipo B11 vs Equipo C2--> Sabado 12
Equipo A1 vs Equipo B1--> Domingo 20
Equipo B5 vs Equipo C4--> Lunes 20
Equipo B2 vs Equipo B8--> Sabado 18
Equipo A3 vs Equipo C3--> Domingo 18
Equipo B10 vs Equipo B7--> Sabado 16
Equipo B6 vs Equipo C6--> Domingo 12
Equipo B4 vs Equipo B3--> Domingo 16
Equipo B9 vs Equipo C1--> Viernes 20

Audiencia Total: 6.53 millones
